
# Raport: Predykcja przepustowości ruchu internetowego (Upload)

## 1. Cel eksperymentu
Głównym zadaniem badawczym jest predykcja rzeczywistej maksymalnej przepływności wysyłania danych, określanej za pomocą parametru `bytes_sec` (bajty na sekundę). Monitorowanie i przewidywanie ruchu sieciowego pozwala dostawcom internetu na elastyczne modelowanie przydziału przepustowości i optymalizację węzłów telekomunikacyjnych w oparciu o cykle dobowe klientów.

## 2. Zaproponowane algorytmy do predykcji
Zaproponowano trzy zróżnicowane estymatory regresyjne, stanowiące współczesny branżowy standard do predykcji danych ustrukturyzowanych (tabelarycznych):

### A. Random Forest Regressor (Las Losowy)
*   **Charakterystyka**: Zaawansowany model zespołowy oparty na koncepcji agragacji (bagging). Generuje kilkaset niezależnych drzew decyzyjnych na różnych podzbiorach cech, a ostateczny wynik to ich średnia wartość.
*   **Zasadność**: Algorytm wybitnie weryfikuje się na danych zaszumionych i sieciowych. Świetnie radzi sobie z kolumnami jakościowymi (Kategorycznymi). Nie wymaga również ścisłego skalowania i wybacza brak standaryzacji względem innych cech.
*   **Ograniczenia**: Lasy losowe słabo zachowują się w warunkach silnej ekstrapolacji – nie będą w stanie "wymyślić" przepustowości powyżej maksymalnej podanej podczas treningu. Ponadto zajmują sporo pamięci ze względu na ogromne gabaryty generowanych drzew.
*   **Parametry**: `n_estimators=200` (optymalna liczba drzew powstrzymująca spadki błędu), `max_depth=15` (przycięto, aby zapobiec przeuczeniu / overfittingowi), `min_samples_leaf=5`.

### B. Gradient Boosting Regressor (Wzmocnienie Gradientowe)
*   **Charakterystyka**: Kolejny zestaw zespołowy, jednak zamiast niezależnych drzew używamy tzw. *boosting*. Model trenuje sekwencje słabych jednostek (płytkich drzew), gdzie każde następne drzewo wchodzi w interakcje z błędami wygenerowanymi przez jego poprzednika. 
*   **Zasadność**: Uznaje się go za złoty standard algorytmów na danych tabelarycznych. Z reguły osiąga o włos optymalniejszą sprawność niż Lasy Losowe ze względu na precyzyjną optymalizację po tzw. pochodnej rezyduów.
*   **Ograniczenia**: Jest modelem powolnym z powodu sekwencyjnego uczenia, a zbyt długa nauka, bardzo szybko prowadzi do zgubnego przeuczenia w stosunku do Lasu Losowego.
*   **Parametry**: `n_estimators=200`, `learning_rate=0.05` (rozsądny mały krok korygujących po każdym drzewku zachowujący stabilność), `max_depth=6`, `subsample=0.8` (trening uwzględnia 80% próbki by zachować stochasticzność).

### C. Multi-Layer Perceptron (Gęsta Sieć Neuronowa)
*   **Charakterystyka**: Wielowarstwowa sieć neuronowa *feed-forward*, oparta na warstwach z ukrytymi nieliniowymi zjawiskami aktywacyjnymi sterowanymi wielkimi grafami połączeń matematycznych.
*   **Zasadność**: Architektura pozwala na chłonięcie bardzo wysokopoziomowych abstrakcji co umożliwia odkrywanie w zbiorach takich ukrytych zasad i korelacji, których statystyczne modele połączone liniowo mogły nie wykryć. 
*   **Ograniczenia**: Całkowita "czarna skrzynka" (utrudniona interpretowalność cech jak np. Importance). Oprócz tego wysoce trudny dobór dziesiątek hiperparametrów i czułość na najmniejsze szumy. Wymuszona 100% standaryzacja macierzy `StandarScaler()`.
*   **Parametry**: Redukująca kształt struktura głęboka ukryta w rozmiarach cech: `hidden_layer_sizes=(256, 128, 64)`, siła regularyzacji krzywej kosztów chroniąca sieć przed over-fittingiem w węzłach: `alpha=0.005`, partiami karmiący silnik sieci do szybszej pracy pod minimalizację adam: `batch_size=512`.



## 3. Prezentacja wyników eksperymentu
Kod treningowy, zoptymalizowany m.in. o grupowanie *Cyclical Target Encodingu*, wygenerował gotowy plik z zestawieniem pod metryki R2, Błąd Przeciętny (MAE), Pierwiastek Odchylenia (RMSE) oraz MAP %. Prezentujemy poniżej jego analityczną formę graficzną i numeryczną.




In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

# Pamiętamy z pliku *.py, że 3 modele zostają wrzucane do górnej tabeli 
df_results = pd.read_csv("wyniki_predykcji.csv", nrows=3)

# Oczyszczenie tabeli nagłówkowych (dla porządku danych wyświetlanych)
if "Model" not in df_results.columns:
    df_results.rename(columns={'Unnamed: 0': 'Model', 'R²': 'R2'}, inplace=True)
else:
    df_results.rename(columns={'R²': 'R2'}, inplace=True)

# Formatowanie tabeli
display(df_results.style.format({
    "MAE": "{:,.2f}",
    "RMSE": "{:,.2f}",
    "R2": "{:.4f}",
    "MAPE (%)": "{:.2f}%"
}).hide(axis="index"))


<VSCode.Cell language="markdown">
### Wykresy analityczne
</VSCode.Cell>

<VSCode.Cell language="python">
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Porównanie skuteczności predyktorów przepustowości łącza (bytes_sec)', fontsize=16, fontweight='bold')

# R2 Score Plot
sns.barplot(data=df_results, x='Model', y='R2', ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('Współczynnik Determinacji (R² Score) [im więcej tym lepiej]')
axes[0, 0].set_ylim(0, 1.1)
for i, val in enumerate(df_results['R2']):
    axes[0, 0].text(i, val + 0.02, f'{val:.4f}', ha='center', fontweight='bold')

# MAPE Plot
sns.barplot(data=df_results, x='Model', y='MAPE (%)', ax=axes[0, 1], palette='magma')
axes[0, 1].set_title('Średni Błąd Procentowy (MAPE) [%] [im mniej tym lepiej]')
for i, val in enumerate(df_results['MAPE (%)']):
    axes[0, 1].text(i, val + 0.5, f'{val:.2f}%', ha='center', fontweight='bold')

# MAE Plot
sns.barplot(data=df_results, x='Model', y='MAE', ax=axes[1, 0], palette='crest')
axes[1, 0].set_title('Średni Błąd Bezwzględny (MAE) [bajty/s] [im mniej tym lepiej]')

# RMSE Plot
sns.barplot(data=df_results, x='Model', y='RMSE', ax=axes[1, 1], palette='flare')
axes[1, 1].set_title('Błąd Kwadratowy (RMSE) [bajty/s] [im mniej tym lepiej]')

plt.tight_layout()
plt.show()
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Wnioski 

**1. Bezwzględny Zwycięzca: Modele Drzewiaste (Gradient Boosting i Random Forest)**
Jak widać po wykresach, algorytm **Gradient Boosting** osiągnął absolutny szczyt możliwości dla tych danych, notując wynik R² na poziomie **0.9985**, stanowiący genialną korelację trafień. Random Forest podąża za nim minimalnie z wynikiem 0.994.

**2. Margines i tolerancja błędu w praktyce inżynieryjnej**
Bardzo interesująco prezentuje się metryka MAP%. Las Losowy (Random Forest) osiąga obłędnie niski błąd względny na poziomie raptem **~0.29%**, a wzmocnienie gradientowe ~3.33%. Te predyktory są wyjątkowo dokładne. Co ważne z punktu widzenia projektowego – MAE rzędu 23 tysięcy bajtów brzmi na olbrzymią liczbe (czyli ~23KB/s) jednak gdy weźmiemy pod uwage łącza klientów pędzące po kilkadziesiąt/set Megabajtów to te szacowane straty odchyłów wykazanych w RMSE stanowią dla operatora promil pomyłki statystycznej z zachowaniem idealnej proporcji.

**3. Ograniczenia dla Sieci Neuronowej (MLP)**
Co typowe dla środowisk tabelarycznych, prosta głęboka sieć Multi-Layer Perceptron wypada znacząco gorzej pod względem stabilności (MAPE ok ~22% oraz R² ok. 0.87) gubiąc korelację w brudnych pikach danych sieciowych. Weryfikuje to branżowe standardy: do ustrukturalizowanych logów infrastruktury (telemetrycznych po cechach numerycznych bez obrazu/tekstów) modele oparte na ensemble'ach drzewiastych są wciąż liderem Data Science i predykcji analitycznej.
</VSCode.Cell>